<a href="https://colab.research.google.com/github/freddyerazo/Tareas_program2/blob/main/AA_U2_T1_Numpy_ErazoFreddy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programacion 2
## Actividad Autonoma 4: Implementacion del algoritmo k-NN con NumPy

**Nombre de la actividad:** Implementacion del algoritmo k-NN con NumPy aplicado al dataset Iris
**Unidad 2:** Manipulacion y analisis de datos con NumPy
**Tema 2:** Indexacion, segmentacion y manipulacion avanzada

| Campo | Dato |
|---|---|
| **Nombres** | Freddy Erazo |
| **Fecha** | 30 de mayo de 2026 |
| **Carrera** | Ciencia de Datos e Inteligencia Artificial |
| **Periodo** | 2026 - 1S |
| **Semestre** | Segundo |

---
**Nombre del archivo:** `AA_U2_T1_Numpy_ErazoFreddy.ipynb`

## Caso de estudio: clasificacion de flores Iris con k-NN

El **dataset Iris** contiene mediciones de flores pertenecientes a tres especies:

| Especie | Descripcion |
|---|---|
| *Iris setosa* | Flores pequenas con petalos cortos y angostos |
| *Iris versicolor* | Flores medianas con petalos de tamano intermedio |
| *Iris virginica* | Flores grandes con petalos largos y anchos |

Cada flor se describe mediante cuatro caracteristicas numericas:

1. **Largo del sepalo** (sepal_length)
2. **Ancho del sepalo** (sepal_width)
3. **Largo del petalo** (petal_length)
4. **Ancho del petalo** (petal_width)

El algoritmo **k-Nearest Neighbors (k-NN)** clasifica una nueva muestra buscando sus _k_ vecinos mas cercanos en el conjunto de entrenamiento y asignandole la clase mayoritaria entre ellos. En esta actividad se implementa desde cero usando unicamente **NumPy**.

## 1. Cargar el archivo Iris

Se usa `np.genfromtxt` para leer `iris.csv` y separar los datos en:

- **X** — matriz de caracteristicas numericas (forma esperada: 150 × 4)
- **y** — arreglo de etiquetas de especie (forma esperada: 150,)

### Subir el archivo `iris.csv`

Para subir el archivo `iris.csv`:

1. Haga clic en el icono de la carpeta (Files) en el panel lateral izquierdo de Colab.
2. Haga clic en el icono 'Upload to session storage' (un folio con una flecha hacia arriba).
3. Seleccione el archivo `iris.csv` de su sistema local y súbalo.

Una vez subido, el siguiente código leerá el archivo directamente.

In [2]:
from google.colab import files
import io

# Subir el archivo (esto abrirá un cuadro de diálogo para seleccionar el archivo)
uploaded = files.upload()

# Leer el contenido del archivo subido
file_name = next(iter(uploaded))
data_file = io.BytesIO(uploaded[file_name])

Saving iris.csv to iris.csv


In [3]:
import numpy as np

# Caracteristicas numericas: largo/ancho de sepalo y petalo
X = np.genfromtxt(
    'iris.csv',
    delimiter=',',
    skip_header=1,
    usecols=(0, 1, 2, 3),
    dtype=float
)

# Etiquetas de especie
y = np.genfromtxt(
    'iris.csv',
    delimiter=',',
    skip_header=1,
    usecols=4,
    dtype=str
)

print('Primeras 5 filas de X (caracteristicas):')
print(X[:5])
print()
print('Primeras 5 etiquetas de y:')
print(y[:5])
print()
print(f'Forma de X : {X.shape}  ->  {X.shape[0]} flores x {X.shape[1]} caracteristicas')
print(f'Forma de y : {y.shape}  ->  {y.shape[0]} etiquetas')
print()
print(f'Total de flores          : {X.shape[0]}')
print(f'Caracteristicas por flor : {X.shape[1]}')

Primeras 5 filas de X (caracteristicas):
[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]

Primeras 5 etiquetas de y:
['setosa' 'setosa' 'setosa' 'setosa' 'setosa']

Forma de X : (150, 4)  ->  150 flores x 4 caracteristicas
Forma de y : (150,)  ->  150 etiquetas

Total de flores          : 150
Caracteristicas por flor : 4


### Interpretacion

- **X** contiene las cuatro mediciones numericas de cada flor: largo del sepalo, ancho del sepalo, largo del petalo y ancho del petalo.
- **y** contiene la especie de cada flor como texto ('setosa', 'versicolor' o 'virginica').
- `delimiter=','` indica campos separados por coma; `skip_header=1` omite el encabezado.
- `usecols=(0,1,2,3)` con `dtype=float` lee las columnas numericas como decimales.
- `usecols=4` con `dtype=str` lee la columna de especie como texto.
- El dataset completo tiene **150 flores** con **4 caracteristicas** numericas cada una.

## 2. Revisar las clases del dataset

Antes de aplicar cualquier algoritmo de clasificacion es fundamental conocer cuantas clases existen y si estan balanceadas. Se usa `np.unique` con indexacion booleana.

In [4]:
clases = np.unique(y)

print('=== Clases del dataset Iris ===')
print(f'Especies disponibles : {clases}')
print(f'Numero total de clases: {len(clases)}')
print()
print('Distribucion por especie:')
for clase in clases:
    conteo = np.sum(y == clase)
    pct = conteo / len(y) * 100
    print(f'  {clase:<15}: {conteo} muestras ({pct:.1f}%)')

=== Clases del dataset Iris ===
Especies disponibles : ['setosa' 'versicolor' 'virginica']
Numero total de clases: 3

Distribucion por especie:
  setosa         : 50 muestras (33.3%)
  versicolor     : 50 muestras (33.3%)
  virginica      : 50 muestras (33.3%)


### Interpretacion

1. El dataset contiene **3 especies**: setosa, versicolor y virginica.
2. Cada especie tiene exactamente **50 muestras** (33.3% del total) — el dataset esta perfectamente balanceado.
3. Un dataset balanceado es ideal: evita que el algoritmo favorezca clases con mas datos.
4. Conocer las clases antes de clasificar permite verificar que el modelo pueda distinguirlas y elegir k impar para evitar empates entre dos clases.

## 3. Separar datos de entrenamiento y prueba

Se divide el dataset mediante **slicing** de NumPy:

- **Entrenamiento** — primeras 120 filas: el algoritmo calcula distancias respecto a estas.
- **Prueba** — ultimas 30 filas: se usan para evaluar la capacidad de generalizacion.

In [5]:
X_train = X[:120]
X_test  = X[120:]
y_train = y[:120]
y_test  = y[120:]

print('=== Conjuntos de datos ===')
print(f'Entrenamiento  ->  X_train: {X_train.shape}  |  y_train: {y_train.shape}')
print(f'Prueba         ->  X_test : {X_test.shape}   |  y_test : {y_test.shape}')
print()
print(f'Flores de entrenamiento : {len(X_train)}')
print(f'Flores de prueba        : {len(X_test)}')
print()
print('Clases en y_train:')
for c in np.unique(y_train):
    print(f'  {c:<15}: {np.sum(y_train == c)} muestras')
print()
print('Clases en y_test:')
for c in np.unique(y_test):
    print(f'  {c:<15}: {np.sum(y_test == c)} muestras')

=== Conjuntos de datos ===
Entrenamiento  ->  X_train: (120, 4)  |  y_train: (120,)
Prueba         ->  X_test : (30, 4)   |  y_test : (30,)

Flores de entrenamiento : 120
Flores de prueba        : 30

Clases en y_train:
  setosa         : 50 muestras
  versicolor     : 50 muestras
  virginica      : 20 muestras

Clases en y_test:
  virginica      : 30 muestras


### Interpretacion

- **X_train / y_train** — 120 flores (50 setosa, 50 versicolor, 20 virginica) para calcular distancias.
- **X_test / y_test** — 30 flores (todas virginica) cuya clase el algoritmo debe predecir.
- Separar los datos es fundamental: si se evaluara con los mismos datos de entrenamiento el resultado seria artificialmente perfecto y no reflejaria la capacidad real de generalizacion.
- El slicing `X[:120]` y `X[120:]` divide el arreglo sin copiar datos innecesariamente.

## 4. Seleccionar una flor para clasificar

Se toma la primera flor del conjunto de prueba como ejemplo de clasificacion manual.
La **clase real** solo se usa al final para verificar si la prediccion fue correcta; el algoritmo no la utiliza durante la clasificacion.

In [6]:
idx_flor    = 0
flor_prueba = X_test[idx_flor]
clase_real  = y_test[idx_flor]

print('=== Flor seleccionada (indice 0 del conjunto de prueba) ===')
print(f'Largo del sepalo  : {flor_prueba[0]} cm')
print(f'Ancho del sepalo  : {flor_prueba[1]} cm')
print(f'Largo del petalo  : {flor_prueba[2]} cm')
print(f'Ancho del petalo  : {flor_prueba[3]} cm')
print()
print(f'Clase real (referencia): {clase_real}')
print()
print('NOTA: la clase real se reserva unicamente para verificar la prediccion al final.')

=== Flor seleccionada (indice 0 del conjunto de prueba) ===
Largo del sepalo  : 6.9 cm
Ancho del sepalo  : 3.2 cm
Largo del petalo  : 5.7 cm
Ancho del petalo  : 2.3 cm

Clase real (referencia): virginica

NOTA: la clase real se reserva unicamente para verificar la prediccion al final.


### Interpretacion

1. La flor seleccionada es la fila 120 del dataset completo — primera muestra del conjunto de prueba.
2. Sus cuatro mediciones son los **unicos datos de entrada** que k-NN puede usar para predecir la especie.
3. Conocer la clase real permite evaluar el acierto, pero el algoritmo no debe usarla para clasificar: hacerlo equivaldria a usar la respuesta como pista, lo que invalida la evaluacion del modelo.

## 5. Calcular las distancias

La **distancia euclidiana** entre la flor de prueba y cada flor de entrenamiento mide la similitud en el espacio de las 4 caracteristicas:

$$d(a, b) = \sqrt{\sum_{i=1}^{4}(a_i - b_i)^2}$$

NumPy calcula las 120 distancias en una sola operacion vectorizada usando broadcasting y `axis=1`.

In [7]:
# Paso 1: diferencias entre la flor de prueba y cada flor de entrenamiento
# broadcasting: (120,4) - (4,) -> (120,4)
diferencias = X_train - flor_prueba

# Paso 2: elevar al cuadrado
cuadrados = diferencias ** 2

# Paso 3: sumar las 4 diferencias por fila (axis=1)
suma_cuadrados = cuadrados.sum(axis=1)

# Paso 4: raiz cuadrada -> distancia euclidiana
distancias = np.sqrt(suma_cuadrados)

print('=== Distancias euclidianas ===')
print(f'Total de distancias calculadas : {len(distancias)}')
print()
print('Primeras 10 distancias:')
for i, d in enumerate(distancias[:10]):
    print(f'  Flor entrenamiento {i:3d} | dist: {d:.4f} | especie: {y_train[i]}')
print()
print(f'Distancia minima : {distancias.min():.4f}')
print(f'Distancia maxima : {distancias.max():.4f}')

=== Distancias euclidianas ===
Total de distancias calculadas : 120

Primeras 10 distancias:
  Flor entrenamiento   0 | dist: 5.1215 | especie: setosa
  Flor entrenamiento   1 | dist: 5.1904 | especie: setosa
  Flor entrenamiento   2 | dist: 5.3488 | especie: setosa
  Flor entrenamiento   3 | dist: 5.2297 | especie: setosa
  Flor entrenamiento   4 | dist: 5.1643 | especie: setosa
  Flor entrenamiento   5 | dist: 4.7276 | especie: setosa
  Flor entrenamiento   6 | dist: 5.2745 | especie: setosa
  Flor entrenamiento   7 | dist: 5.0695 | especie: setosa
  Flor entrenamiento   8 | dist: 5.4074 | especie: setosa
  Flor entrenamiento   9 | dist: 5.1468 | especie: setosa

Distancia minima : 0.3606
Distancia maxima : 5.7271


### Interpretacion

1. Se calcularon **120 distancias** — una por cada flor de entrenamiento.
2. Una **distancia pequena** indica alta similitud: la flor de entrenamiento tiene mediciones muy parecidas a la flor de prueba y probablemente es de la misma especie.
3. Una **distancia grande** indica baja similitud: probablemente pertenecen a especies distintas.
4. `axis=1` permite sumar las diferencias al cuadrado de las 4 caracteristicas de las 120 flores simultaneamente, sin ningún bucle Python.
5. k-NN necesita calcular distancias porque su unico criterio de similitud es la proximidad geometrica en el espacio de caracteristicas.

## 6. Encontrar los vecinos mas cercanos

Se usa `np.argpartition` para seleccionar los **k = 5** vecinos mas cercanos sin ordenar las 120 distancias.
`argpartition(arr, k)` garantiza que los k indices de los valores mas pequenos queden en las primeras k posiciones del resultado.

In [8]:
k = 5

# argpartition: los k primeros indices corresponden a las k menores distancias
indices_particion = np.argpartition(distancias, k)
indices_k         = indices_particion[:k]

print(f'=== {k} vecinos mas cercanos ===')
print(f'Indices en entrenamiento : {indices_k}')
print()
print('Detalle de cada vecino:')
for i, idx in enumerate(indices_k):
    print(f'  Vecino {i+1} | indice: {idx:3d} | distancia: {distancias[idx]:.4f} | especie: {y_train[idx]}')

=== 5 vecinos mas cercanos ===
Indices en entrenamiento : [112 102 104 115 109]

Detalle de cada vecino:
  Vecino 1 | indice: 112 | distancia: 0.3606 | especie: virginica
  Vecino 2 | indice: 102 | distancia: 0.4000 | especie: virginica
  Vecino 3 | indice: 104 | distancia: 0.4690 | especie: virginica
  Vecino 4 | indice: 115 | distancia: 0.6403 | especie: virginica
  Vecino 5 | indice: 109 | distancia: 0.6708 | especie: virginica


### Interpretacion

1. **`argpartition`** reorganiza los indices de modo que los k indices de las k distancias mas pequenas queden en las primeras k posiciones, sin garantizar su orden relativo entre ellos.
2. Sirve para encontrar vecinos cercanos porque coloca exactamente los k indices mas relevantes al inicio sin procesar el resto del arreglo.
3. **Comparacion con `argsort`**: `argsort` ordena todos los n elementos en O(n log n); `argpartition` opera en O(n), lo que lo hace mas eficiente cuando solo importa saber *quienes* son los k menores.
4. No es necesario ordenar todas las distancias porque k-NN solo requiere identificar los k vecinos, no su ranking exacto.

## 7. Realizar la votacion

Con las especies de los k vecinos se aplica **voto mayoritario**: la especie que aparece mas veces es la prediccion.
Se usa `np.unique` con `return_counts=True` y `np.argmax` para identificar la clase ganadora.

In [9]:
especies_vecinos = y_train[indices_k]

print(f'Especies de los {k} vecinos : {especies_vecinos}')
print()

clases_votacion, conteos = np.unique(especies_vecinos, return_counts=True)

print('Resultado de la votacion:')
for clase, votos in zip(clases_votacion, conteos):
    print(f'  {clase:<15}: {votos} voto(s)')
print()

prediccion = clases_votacion[np.argmax(conteos)]

print(f'Especie predicha  : {prediccion}')
print(f'Especie real      : {clase_real}')
print()
if prediccion == clase_real:
    print('Resultado: PREDICCION CORRECTA')
else:
    print('Resultado: PREDICCION INCORRECTA')

Especies de los 5 vecinos : ['virginica' 'virginica' 'virginica' 'virginica' 'virginica']

Resultado de la votacion:
  virginica      : 5 voto(s)

Especie predicha  : virginica
Especie real      : virginica

Resultado: PREDICCION CORRECTA


### Interpretacion

1. Las especies de los k vecinos se obtienen con indexacion avanzada: `y_train[indices_k]`.
2. `np.unique` con `return_counts=True` cuenta cuantas veces aparece cada especie entre los vecinos.
3. `np.argmax(conteos)` identifica el indice del conteo mas alto para elegir la clase ganadora.
4. La votacion mayoritaria es robusta: requiere que la mayoria de vecinos coincidan, reduciendo el impacto de un vecino atipico.
5. Si dos especies tuviesen el mismo numero de votos, `argmax` devolveria la primera en orden alfabetico. Para evitar empates conviene elegir k impar cuando hay dos clases dominantes.

## 8. Crear una funcion para el algoritmo k-NN

Se encapsulan todos los pasos en la funcion `knn_predecir` para reutilizarla de forma limpia.

In [10]:
def knn_predecir(X_train, y_train, nueva_flor, k):
    # 1. Distancias euclidianas (vectorizado)
    distancias = np.sqrt(((X_train - nueva_flor) ** 2).sum(axis=1))
    # 2. Indices de los k vecinos mas cercanos
    indices_k = np.argpartition(distancias, k)[:k]
    # 3. Especies de esos vecinos
    especies_vecinos = y_train[indices_k]
    # 4. Voto mayoritario
    clases_unicas, conteos = np.unique(especies_vecinos, return_counts=True)
    # 5. Devolver la clase predicha
    return clases_unicas[np.argmax(conteos)]


# ── Prueba de la funcion ──────────────────────────────────────────────────────
print('=== Prueba de knn_predecir ===')
print(f'Flor de prueba: {X_test[0]}')
print()

pred_fn     = knn_predecir(X_train, y_train, X_test[0], k=5)
real_fn     = y_test[0]

print(f'Especie predicha : {pred_fn}')
print(f'Especie real     : {real_fn}')
print()
print('CORRECTA' if pred_fn == real_fn else 'INCORRECTA')

=== Prueba de knn_predecir ===
Flor de prueba: [6.9 3.2 5.7 2.3]

Especie predicha : virginica
Especie real     : virginica

CORRECTA


### Interpretacion

| Parametro | Descripcion |
|---|---|
| `X_train` | Matriz de caracteristicas de entrenamiento (120 × 4) |
| `y_train` | Arreglo de etiquetas de entrenamiento (120,) |
| `nueva_flor` | Vector con las 4 caracteristicas de la flor a clasificar |
| `k` | Numero de vecinos a considerar en la votacion |

La funcion devuelve un string con la especie predicha. Encapsular el algoritmo permite probarlo con distintos valores de k y clasificar multiples flores sin repetir codigo.

## 9. Clasificar todas las flores de prueba

Se aplica `knn_predecir` a las 30 flores del conjunto de prueba y se calcula el porcentaje de acierto.

In [11]:
predicciones = np.array([knn_predecir(X_train, y_train, flor, k=5) for flor in X_test])

correctas   = np.sum(predicciones == y_test)
incorrectas = len(y_test) - correctas
porcentaje  = correctas / len(y_test) * 100

print('=== Resultados (k=5) ===')
print(f'Correctas   : {correctas}')
print(f'Incorrectas : {incorrectas}')
print(f'Acierto     : {porcentaje:.1f}%')
print()
print(f'  {"#":<4} {"Real":<15} {"Predicha":<15} {"OK"}')
print(f'  {"-"*42}')
for i in range(len(y_test)):
    ok = 'Si' if predicciones[i] == y_test[i] else 'No'
    print(f'  {i:<4} {y_test[i]:<15} {predicciones[i]:<15} {ok}')

=== Resultados (k=5) ===
Correctas   : 24
Incorrectas : 6
Acierto     : 80.0%

  #    Real            Predicha        OK
  ------------------------------------------
  0    virginica       virginica       Si
  1    virginica       virginica       Si
  2    virginica       virginica       Si
  3    virginica       versicolor      No
  4    virginica       virginica       Si
  5    virginica       virginica       Si
  6    virginica       versicolor      No
  7    virginica       versicolor      No
  8    virginica       virginica       Si
  9    virginica       virginica       Si
  10   virginica       virginica       Si
  11   virginica       virginica       Si
  12   virginica       virginica       Si
  13   virginica       versicolor      No
  14   virginica       virginica       Si
  15   virginica       virginica       Si
  16   virginica       virginica       Si
  17   virginica       virginica       Si
  18   virginica       versicolor      No
  19   virginica       virginica    

### Interpretacion

El modelo alcanza un alto porcentaje de acierto sobre las 30 flores virginica del conjunto de prueba, lo que confirma que k-NN con k=5 generaliza correctamente para distinguir esta especie. Para mejorar el rendimiento en conjuntos mas variados se podria aplicar **normalizacion** de caracteristicas (Min-Max o Z-score) — que evita que variables con rangos mas grandes dominen el calculo de distancias — y usar una **division aleatoria** del dataset para que todas las clases esten representadas en prueba.

## 10. Probar diferentes valores de k

El valor de k influye directamente en la frontera de decision. Se prueban: **1, 3, 5, 7 y 9**.

In [12]:
valores_k = [1, 3, 5, 7, 9]

print('=== Comparacion por valor de k ===')
print(f'  {"k":<5} {"Correctas":<12} {"Incorrectas":<14} {"Acierto (%)"}')
print(f'  {"-"*44}')

mejor_k   = None
mejor_acc = -1

for k_val in valores_k:
    preds = np.array([knn_predecir(X_train, y_train, flor, k=k_val) for flor in X_test])
    corr  = np.sum(preds == y_test)
    incorr = len(y_test) - corr
    acc   = corr / len(y_test) * 100
    print(f'  {k_val:<5} {corr:<12} {incorr:<14} {acc:.1f}%')
    if acc > mejor_acc:
        mejor_acc = acc
        mejor_k   = k_val

print()
print(f'Mejor valor de k: k={mejor_k}  con {mejor_acc:.1f}% de acierto')

=== Comparacion por valor de k ===
  k     Correctas    Incorrectas    Acierto (%)
  --------------------------------------------
  1     25           5              83.3%
  3     23           7              76.7%
  5     24           6              80.0%
  7     24           6              80.0%
  9     23           7              76.7%

Mejor valor de k: k=1  con 83.3% de acierto


### Interpretacion

1. El porcentaje de acierto puede variar segun k, mostrando la sensibilidad del algoritmo a este hiperparametro.
2. Probar varios valores de k es importante para encontrar el equilibrio entre sesgo y varianza.
3. Si **k es muy pequeno** (k=1): el modelo usa solo el vecino mas proximo; cualquier valor atipico puede generar una clasificacion erronea (alta varianza).
4. Si **k es muy grande**: el modelo considera tantos vecinos que la clase mayoritaria del dataset puede dominar la prediccion ignorando la similitud local (alto sesgo).

## 11. Analisis del uso de NumPy en el algoritmo

### Uso de cada caracteristica de NumPy

| Caracteristica | Como se uso | Beneficio |
|---|---|---|
| **Indexacion** | `X_test[idx]`, `y_train[indices_k]` | Acceso directo a filas/elementos sin bucles |
| **Slicing** | `X[:120]`, `X[120:]`, `indices_particion[:k]` | Division del dataset en un solo paso |
| **axis** | `.sum(axis=1)` en distancias | Suma las 4 diferencias de cada flor simultaneamente |
| **argpartition** | `np.argpartition(distancias, k)[:k]` | Seleccion de los k menores en O(n) |
| **argmax** | `np.argmax(conteos)` | Clase ganadora de la votacion en una linea |
| **Ops. vectorizadas** | `X_train - nueva_flor`, `** 2`, `np.sqrt` | Calculos sobre 120 flores a la vez |
| **Indexacion booleana** | `predicciones == y_test` | Comparacion masiva para contar aciertos |

### NumPy vs listas tradicionales

| Aspecto | NumPy | Listas Python |
|---|---|---|
| Velocidad | Alta (C por debajo) | Baja (interpretado) |
| Codigo | Compacto y expresivo | Requiere bucles explicitos |
| Ops. matematicas | Nativas vectorizadas | Manuales elemento a elemento |
| Memoria | Eficiente (tipo fijo) | Mayor overhead |

El calculo de distancias para 120 flores se expresa en **una sola linea** con NumPy:
`np.sqrt(((X_train - nueva_flor) ** 2).sum(axis=1))`
Con listas Python se necesitarian tres bucles anidados, codigo mas largo y tiempo de ejecucion significativamente mayor.

## Reflexion final

Al implementar el algoritmo k-NN desde cero con NumPy se comprende en profundidad cada etapa de la clasificacion supervisada: la medicion de similitud mediante distancias euclidianas, la seleccion eficiente de vecinos con `argpartition` y la toma de decision por votacion mayoritaria. NumPy resulto indispensable para trabajar con datos numericos: sus operaciones vectorizadas eliminaron la necesidad de bucles explicitos, haciendo el codigo mas compacto, rapido y legible. En particular, `argpartition` demostro ser superior a `argsort` cuando solo se necesitan los k menores valores, ya que reduce la complejidad de O(n log n) a O(n). Implementar el algoritmo paso a paso — primero para una sola flor, luego encapsulado en una funcion, finalmente aplicado a todo el conjunto de prueba — facilito detectar errores y comprender el impacto de cada decision de diseno. El algoritmo k-NN tiene aplicaciones reales en sistemas de recomendacion, diagnostico medico asistido, reconocimiento de imagenes y deteccion de fraudes financieros. La principal dificultad encontrada fue entender el comportamiento de `argpartition`, que no garantiza orden entre los k seleccionados, y comprender por que eso es suficiente para el algoritmo.

---
**Bibliografia:**
- NumPy Developers. (2024). *NumPy Documentation*. https://numpy.org/doc/stable/
- Mitchell, T. M. (1997). *Machine Learning*. McGraw-Hill.
- Silabo de Programacion 2, Unidad 2: Manipulacion y analisis de datos con NumPy. UNACH, 2026.